# 01 — Data Preprocessing
**Project:** Which Features Are Most Predictive of Hiring Outcomes: Education, Experience, or Skills?  
**Course:** Machine Learning (Year 2 — Data Analytics)  
**Dataset:** 70K Job Applicants HR Dataset — https://www.kaggle.com/datasets/ayushtankha/70k-job-applicants-data-human-resource

---

## What this notebook does
This notebook prepares the dataset for our four experiments:
1. **Education-only** features vs. hiring outcome
2. **Experience-only** features vs. hiring outcome
3. **Skills-only** features vs. hiring outcome
4. **Combined** (all features) vs. hiring outcome

For each experiment we produce a stratified 80/20 train/test split and save the results to `data/processed/`.

### 🚀 Getting the Data (do this first!)
The raw data is **not committed to this repo**. Before running this notebook, download the dataset with one command:
```bash
python data/download_data.py
```
This script uses the Kaggle API to download and unzip the dataset into `data/raw/`.  
If you don't have your `kaggle.json` API key set up yet, the script will walk you through it.  
See `data/README.md` for full details.

### ⚠️ Data Leakage Note
Only cleaned, processed CSVs live in `data/processed/` (they contain no raw PII and are small enough to commit).

We are also careful to avoid **feature leakage**: every feature we use is one that would be available *before* a hiring decision is made. `HiringDecision` is our target variable only — it never appears as a feature.

---
## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Make sure the processed output folder exists
os.makedirs('../data/processed', exist_ok=True)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('All libraries imported successfully!')

---
## 1. Load the Dataset

The raw CSV should be at `data/raw/job_applicants.csv`.  
If it's not there yet, run **`python data/download_data.py`** from the project root first.

In [ ]:
# Load the raw dataset
RAW_DATA_PATH = '../data/raw/job_applicants.csv'

df = pd.read_csv(RAW_DATA_PATH)

print(f'Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 2. Exploratory Data Analysis (EDA)

Before any cleaning, we want to understand the dataset:
- How many rows and columns do we have?
- What data types are present?
- Are there missing values?
- What does the target variable look like (class balance)?

In [ ]:
# --- Shape and column names ---
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())

In [ ]:
# --- Data types ---
print('Data types:')
print(df.dtypes)

In [ ]:
# --- Basic statistics for numeric columns ---
df.describe()

In [ ]:
# --- Missing values check ---
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

print('Missing values per column:')
print(missing_df[missing_df['Missing Count'] > 0])

if missing_df['Missing Count'].sum() == 0:
    print('No missing values found!')

In [ ]:
# --- Target variable distribution ---
print('Target variable (HiringDecision) distribution:')
print(df['HiringDecision'].value_counts())
print()
print('Class balance (%):')
print(df['HiringDecision'].value_counts(normalize=True).mul(100).round(1))

# Plot it
plt.figure(figsize=(5, 3))
df['HiringDecision'].value_counts().plot(kind='bar', color=['#E74C3C', '#2ECC71'], edgecolor='black')
plt.title('Hiring Decision Distribution')
plt.xlabel('HiringDecision (0 = Not Hired, 1 = Hired)')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- Unique values for categorical columns ---
categorical_cols = df.select_dtypes(include='object').columns.tolist()
print('Categorical columns:', categorical_cols)
print()
for col in categorical_cols:
    print(f'{col}: {df[col].unique()}')

---
## 3. Feature Groups

Our research question is: *Which features are most predictive of hiring outcomes — education, experience, or skills?*

We define four groups:

| Group | Features |
|-------|----------|
| **Education** | `EducationLevel` |
| **Experience** | `ExperienceYears`, `PreviousCompanies` |
| **Skills** | `SkillScore` |
| **Combined** | All of the above + `Age`, `Gender`, `Distance_from_Company`, `InterviewScore`, `AccessToResources`, `AvgSessionDurationMinutes`, `WorkLifeBalance`, `RecruitmentStrategy` |

The **target variable** is `HiringDecision` (1 = Hired, 0 = Not Hired).

In [ ]:
# Define feature groups
TARGET = 'HiringDecision'

FEATURES_EDUCATION   = ['EducationLevel']
FEATURES_EXPERIENCE  = ['ExperienceYears', 'PreviousCompanies']
FEATURES_SKILLS      = ['SkillScore']
FEATURES_COMBINED    = [
    'EducationLevel', 'ExperienceYears', 'PreviousCompanies', 'SkillScore',
    'Age', 'Gender', 'Distance_from_Company', 'InterviewScore',
    'AccessToResources', 'AvgSessionDurationMinutes', 'WorkLifeBalance',
    'RecruitmentStrategy'
]

print('Feature groups defined:')
print(f'  Education:  {FEATURES_EDUCATION}')
print(f'  Experience: {FEATURES_EXPERIENCE}')
print(f'  Skills:     {FEATURES_SKILLS}')
print(f'  Combined:   {FEATURES_COMBINED}')
print(f'  Target:     {TARGET}')

---
## 4. Handling Missing Values

**Strategy:**
- If there are **no missing values** (which we expect for this dataset), no imputation is needed.
- If missing values appear:
  - **Numeric columns** → fill with the **median** (more robust than mean for skewed data).
  - **Categorical columns** → fill with the **mode** (most frequent value).
- We do NOT drop rows unless more than 50% of values in that row are missing.

This strategy keeps our dataset size maximal and is appropriate for a classification task with a relatively balanced dataset.

In [ ]:
df_clean = df.copy()  # Always work on a copy to preserve original

# Separate columns by type
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_clean.select_dtypes(include='object').columns.tolist()

# Remove the target from these lists (we never impute the target)
numeric_cols = [c for c in numeric_cols if c != TARGET]

# Fill missing numerics with median
for col in numeric_cols:
    if df_clean[col].isnull().any():
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f'  Filled {col} (numeric) with median = {median_val:.2f}')

# Fill missing categoricals with mode
for col in cat_cols:
    if df_clean[col].isnull().any():
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f'  Filled {col} (categorical) with mode = "{mode_val}"')

# Final check
remaining_missing = df_clean.isnull().sum().sum()
print(f'\nTotal missing values after imputation: {remaining_missing}')
print(f'Rows in clean dataset: {len(df_clean):,}')

---
## 5. Encoding Categorical Variables

Machine learning models require **numeric input**. Our categorical columns need to be converted.

**Encoding choices:**

| Column | Encoding | Reason |
|--------|----------|--------|
| `EducationLevel` | **Label Encoding** | This is an *ordinal* variable — there's a natural order (e.g., High School < Bachelor's < Master's < PhD). Label encoding preserves that order. |
| `Gender` | **Label Encoding** | Binary or few categories — simple and effective for tree-based models. |
| `RecruitmentStrategy` | **One-Hot Encoding** | *Nominal* variable — no natural order between recruitment strategies. One-hot avoids the model thinking "Online > Referral" just because of numeric ordering. |

> **Why not one-hot everything?** One-hot can cause the *curse of dimensionality* with many categories. For ordinal variables, label encoding is actually the better choice.

In [ ]:
df_encoded = df_clean.copy()

# --- Label encode EducationLevel (ordinal) ---
# We define the order manually so the encoding is meaningful
education_order = ['High School', "Bachelor's", "Master's", 'PhD']

# Check what values actually exist in the column
actual_edu_values = df_encoded['EducationLevel'].unique()
print('EducationLevel unique values:', sorted(actual_edu_values))

# If the values match our expected order, use it; otherwise fall back to LabelEncoder
if all(v in education_order for v in actual_edu_values):
    edu_map = {level: i for i, level in enumerate(education_order)}
    df_encoded['EducationLevel'] = df_encoded['EducationLevel'].map(edu_map)
    print('EducationLevel mapped:', edu_map)
else:
    le_edu = LabelEncoder()
    df_encoded['EducationLevel'] = le_edu.fit_transform(df_encoded['EducationLevel'])
    print('EducationLevel label encoded (fallback). Classes:', le_edu.classes_)

In [ ]:
# --- Label encode Gender (binary / few categories) ---
print('Gender unique values:', df_encoded['Gender'].unique())

le_gender = LabelEncoder()
df_encoded['Gender'] = le_gender.fit_transform(df_encoded['Gender'])
print('Gender encoded. Mapping:', dict(zip(le_gender.classes_, le_gender.transform(le_gender.classes_))))

In [ ]:
# --- One-hot encode RecruitmentStrategy (nominal) ---
print('RecruitmentStrategy unique values:', df_encoded['RecruitmentStrategy'].unique())

df_encoded = pd.get_dummies(df_encoded, columns=['RecruitmentStrategy'], prefix='RecStrat', drop_first=True)
# drop_first=True avoids the "dummy variable trap" (perfect multicollinearity)

print('Columns after one-hot encoding RecruitmentStrategy:')
print([c for c in df_encoded.columns if 'RecStrat' in c])

In [ ]:
# --- Update FEATURES_COMBINED to include new one-hot columns ---
recstrat_cols = [c for c in df_encoded.columns if 'RecStrat' in c]

FEATURES_COMBINED_ENCODED = [
    'EducationLevel', 'ExperienceYears', 'PreviousCompanies', 'SkillScore',
    'Age', 'Gender', 'Distance_from_Company', 'InterviewScore',
    'AccessToResources', 'AvgSessionDurationMinutes', 'WorkLifeBalance'
] + recstrat_cols

print('Final combined feature set:')
print(FEATURES_COMBINED_ENCODED)
print(f'\nTotal features in combined group: {len(FEATURES_COMBINED_ENCODED)}')

In [ ]:
# --- Preview the fully encoded dataset ---
print('Encoded dataset shape:', df_encoded.shape)
df_encoded.head()

---
## 6. Train/Test Split

We split the data into **80% training** and **20% testing** using a **stratified split**.

**Why stratified?**  
Stratified splitting ensures the proportion of `HiringDecision` = 0 and 1 is the same in both the training set and the test set. This is important when there's any class imbalance — without stratification, your test set might accidentally have very few examples of one class.

We set `random_state=42` so our results are **reproducible** — anyone on the team who runs this notebook will get the same split.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE    = 0.20   # 20% goes to test, 80% to train

y = df_encoded[TARGET]

def make_split(feature_cols):
    """Helper: given a list of feature column names, return X_train, X_test, y_train, y_test."""
    X = df_encoded[feature_cols]
    return train_test_split(
        X, y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_STATE
    )

# --- Education ---
X_train_edu, X_test_edu, y_train, y_test = make_split(FEATURES_EDUCATION)

# --- Experience ---
X_train_exp, X_test_exp, _, _ = make_split(FEATURES_EXPERIENCE)

# --- Skills ---
X_train_skl, X_test_skl, _, _ = make_split(FEATURES_SKILLS)

# --- Combined ---
X_train_comb, X_test_comb, _, _ = make_split(FEATURES_COMBINED_ENCODED)

print('Split sizes (using education split as representative):')
print(f'  Train: {len(X_train_edu):,} rows  |  Test: {len(X_test_edu):,} rows')
print()
print('Class distribution in training set:')
print(y_train.value_counts(normalize=True).mul(100).round(1))
print()
print('Class distribution in test set:')
print(y_test.value_counts(normalize=True).mul(100).round(1))

---
## 7. Save Processed Data

We save all four train/test splits to `data/processed/`. These files **are** committed to the repo — they are clean, numeric, and contain no raw/sensitive data.

We include `y_train` / `y_test` alongside each feature split so downstream notebooks can load everything from one place without needing to re-split.

In [ ]:
PROCESSED_DIR = '../data/processed'
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Helper: save train and test splits with the target column included
def save_split(X_train, X_test, y_train, y_test, name):
    train_df = X_train.copy()
    train_df[TARGET] = y_train.values
    
    test_df = X_test.copy()
    test_df[TARGET] = y_test.values
    
    train_path = os.path.join(PROCESSED_DIR, f'train_{name}.csv')
    test_path  = os.path.join(PROCESSED_DIR, f'test_{name}.csv')
    
    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path,  index=False)
    
    print(f'Saved: {train_path}  ({len(train_df):,} rows)')
    print(f'Saved: {test_path}   ({len(test_df):,} rows)')

# Save all four experiments
save_split(X_train_edu,  X_test_edu,  y_train, y_test, 'education')
save_split(X_train_exp,  X_test_exp,  y_train, y_test, 'experience')
save_split(X_train_skl,  X_test_skl,  y_train, y_test, 'skills')
save_split(X_train_comb, X_test_comb, y_train, y_test, 'combined')

print('\nAll processed files saved to data/processed/')

---
## 8. Quick Sanity Checks

Let's do a final check to make sure everything looks right before handing off to the modelling notebooks.

In [ ]:
# List all saved files
saved_files = os.listdir(PROCESSED_DIR)
print('Files in data/processed/:')
for f in sorted(saved_files):
    path = os.path.join(PROCESSED_DIR, f)
    size_kb = os.path.getsize(path) / 1024
    print(f'  {f}  ({size_kb:.1f} KB)')

In [ ]:
# Quick peek at the combined training set
train_combined = pd.read_csv(os.path.join(PROCESSED_DIR, 'train_combined.csv'))
print('Combined train set shape:', train_combined.shape)
train_combined.head()

In [ ]:
# Verify no missing values in any processed file
all_ok = True
for f in sorted(saved_files):
    temp = pd.read_csv(os.path.join(PROCESSED_DIR, f))
    missing = temp.isnull().sum().sum()
    status = '✓' if missing == 0 else f'✗ ({missing} missing)'
    print(f'  {f}: {status}')
    if missing > 0:
        all_ok = False

print()
if all_ok:
    print('All processed files are clean — no missing values!')
else:
    print('WARNING: Some files still have missing values. Review the imputation step.')

---
## 9. Feature Distributions (Quick Visual Check)

A final visual check to understand the key numeric features before modelling.

In [ ]:
# Plot distributions of key numeric features, coloured by hiring outcome
key_features = ['EducationLevel', 'ExperienceYears', 'SkillScore', 'InterviewScore']

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    for outcome, label, color in [(0, 'Not Hired', '#E74C3C'), (1, 'Hired', '#2ECC71')]:
        subset = df_encoded[df_encoded[TARGET] == outcome][feat]
        axes[i].hist(subset, bins=20, alpha=0.6, label=label, color=color, edgecolor='black')
    axes[i].set_title(f'{feat} by Hiring Decision')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Count')
    axes[i].legend()

plt.suptitle('Feature Distributions by Hiring Outcome', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Summary

This notebook has:

1. ✅ Loaded the 70K job applicants dataset from `data/raw/`
2. ✅ Explored the data (shape, dtypes, missing values, class balance)
3. ✅ Defined four feature groups (Education, Experience, Skills, Combined)
4. ✅ Handled missing values (median for numeric, mode for categorical)
5. ✅ Encoded categorical variables (ordinal label encoding for EducationLevel, one-hot for RecruitmentStrategy)
6. ✅ Created stratified 80/20 train/test splits with `random_state=42`
7. ✅ Saved 8 cleaned CSV files to `data/processed/`

**Next steps:** Use these processed files in the modelling notebooks (one per experiment group).

---
*Tip for teammates: You only need to run this notebook once after downloading the raw data. The processed files are already committed to the repo so you can skip straight to the modelling notebooks if they already exist.*